<a href="https://colab.research.google.com/github/XTMay/ML_DL/blob/main/NLP/IntelligentStopwordsManager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import jieba
import jieba.posseg as pseg
import jieba.analyse
from typing import List, Tuple, Dict, Set
from collections import defaultdict, Counter
import numpy as np
import os


class AdvancedChineseTokenizer:
    """
    高级中文分词器
    集成多种分词模式和自定义功能
    """
    def __init__(self, user_dict_path: str = None, stop_words_path: str = None):
        """初始化分词器"""
        # 加载用户词典
        if user_dict_path and os.path.exists(user_dict_path):
            jieba.load_userdict(user_dict_path)
            print(f"已加载用户词典: {user_dict_path}")

        # 加载停用词
        self.stop_words = set()
        if stop_words_path and os.path.exists(stop_words_path):
            with open(stop_words_path, 'r', encoding='utf-8') as f:
                self.stop_words = set(f.read().splitlines())
            print(f"已加载停用词: {len(self.stop_words)} 个")

        # 初始化默认停用词
        self._init_default_stop_words()

        # 分词统计信息
        self.stats = {
            'total_texts': 0,
            'total_tokens': 0,
            'avg_tokens_per_text': 0,
            'unique_tokens': set()
        }

    def _init_default_stop_words(self):
        """ 初始化默认停用词列表"""
        default_stop_words = {
            '的', '了', '在', '是', '我', '有', '和', '就', '不', '人', '都',
            '一', '一个', '上', '也', '很', '到', '说', '要', '去', '你', '会',
            '着', '没有', '看', '好', '自己', '这', '那', '什么', '怎么',
            '为什么', '因为', '所以', '但是', '如果', '或者', '以及', '以上',
            '以下', '之前', '之后', '这样', '那样', '这些', '那些', '这里', '那里'
        }
        self.stop_words.update(default_stop_words)

    def precise_cut(self, text: str) -> List[str]:
        """ 精确模式分词"""
        return list(jieba.cut(text, cut_all=False))

    def full_cut(self, text: str) -> List[str]:
        """ 全模式分词"""
        return list(jieba.cut(text, cut_all=True))

    def search_cut(self, text: str) -> List[str]:
        """ 搜索引擎模式分词"""
        return list(jieba.cut_for_search(text))

    def pos_tagging(self, text: str) -> List[Tuple[str, str]]:
        """ 词性标注 """
        return [(word, pos) for word, pos in pseg.cut(text)]

    def extract_keywords(self, text: str, topK: int = 10, method: str = 'tfidf') -> List[Tuple[str, float]]:
        """ 关键词提取 """
        if method == 'tfidf':
            return jieba.analyse.extract_tags(text, topK=topK, withWeight=True)
        elif method == 'textrank':
            return jieba.analyse.textrank(text, topK=topK, withWeight=True)
        else:
            raise ValueError("method must be 'tfidf' or 'textrank'")

    def filter_by_pos(self, text: str, keep_pos: Set[str] = None) -> List[str]:
        """ 根据词性过滤词汇 """
        if keep_pos is None:
            keep_pos = {'n', 'nr', 'ns', 'nt', 'nz', 'v', 'vd', 'vn', 'a', 'ad'}
        words_pos = self.pos_tagging(text)
        return [word for word, pos in words_pos if any(pos.startswith(p) for p in keep_pos)]

    def remove_stop_words(self, words: List[str]) -> List[str]:
        """ 移除停用词 """
        return [word for word in words if word not in self.stop_words and len(word) > 1]

    def add_user_word(self, word: str, freq: int = None, tag: str = None):
        """ 添加用户自定义词汇 """
        if freq and tag:
            jieba.add_word(word, freq, tag)
        elif freq:
            jieba.add_word(word, freq)
        else:
            jieba.add_word(word)
        print(f"已添加用户词汇: {word}")

    def delete_user_word(self, word: str):
        """ 删除用户自定义词汇 """
        jieba.del_word(word)
        print(f"已删除用户词汇: {word}")

    def suggest_freq(self, segment: str, tune: bool = False) -> int:
        """ 调节词汇频率 """
        freq = jieba.suggest_freq(segment, tune)
        print(f"词汇 '{segment}' 的建议频率: {freq}")
        return freq

    def tokenize_with_options(self, text: str, options: Dict) -> List[str]:
        """ 带选项的分词 """
        mode = options.get('mode', 'precise')
        if mode == 'precise':
            words = self.precise_cut(text)
        elif mode == 'full':
            words = self.full_cut(text)
        elif mode == 'search':
            words = self.search_cut(text)
        else:
            words = self.precise_cut(text)

        if options.get('filter_pos', False):
            keep_pos = options.get('keep_pos', None)
            words = self.filter_by_pos(text, keep_pos)

        if options.get('remove_stop_words', True):
            words = self.remove_stop_words(words)

        min_length = options.get('min_length', 1)
        words = [word for word in words if len(word) >= min_length]

        self._update_stats(words)
        return words

    def _update_stats(self, words: List[str]):
        self.stats['total_texts'] += 1
        self.stats['total_tokens'] += len(words)
        self.stats['unique_tokens'].update(words)
        self.stats['avg_tokens_per_text'] = self.stats['total_tokens'] / self.stats['total_texts']

    def get_tokenization_stats(self) -> Dict:
        stats = self.stats.copy()
        stats['unique_tokens_count'] = len(stats['unique_tokens'])
        stats['vocabulary_size'] = len(stats['unique_tokens'])
        del stats['unique_tokens']
        return stats

    def save_user_dict(self, words: List[str], file_path: str):
        with open(file_path, 'w', encoding='utf-8') as f:
            for word in words:
                f.write(f"{word}\n")
        print(f"已保存用户词典到: {file_path}")

    def analyze_segmentation_quality(self, text: str) -> Dict:
        precise_words = self.precise_cut(text)
        full_words = self.full_cut(text)
        search_words = self.search_cut(text)
        quality_metrics = {
            'text_length': len(text),
            'precise_tokens': len(precise_words),
            'full_tokens': len(full_words),
            'search_tokens': len(search_words),
            'avg_token_length': sum(len(w) for w in precise_words)/len(precise_words) if precise_words else 0,
            'single_char_ratio': sum(1 for w in precise_words if len(w)==1)/len(precise_words) if precise_words else 0,
            'oov_estimation': sum(1 for w in precise_words if w not in jieba.dt.FREQ)/len(precise_words) if precise_words else 0
        }
        return quality_metrics


class IntelligentStopwordsManager:
    """
    智能停用词管理系统
    支持动态停用词发现、领域自适应、效果评估
    """
    def __init__(self, language: str = 'chinese'):
        self.language = language
        self.stopwords = set()
        self.custom_stopwords = set()
        self.domain_stopwords = set()
        self.word_stats = defaultdict(lambda: {'total_freq': 0, 'doc_freq': 0, 'contexts': []})
        self._load_default_stopwords()

    def _load_default_stopwords(self):
        if self.language == 'chinese':
            self.stopwords.update({'的','了','在','是','我','有','和','就','不','人','都'})
        elif self.language == 'english':
            self.stopwords.update({'a','an','and','the','is','in','it','of','to'})

    def add_words(self, words: List[str], word_type: str='custom'):
        if isinstance(words, str):
            words = [words]
        word_set = set(words)
        self.stopwords.update(word_set)
        if word_type == 'custom':
            self.custom_stopwords.update(word_set)
        elif word_type == 'domain':
            self.domain_stopwords.update(word_set)
        print(f"已添加 {len(word_set)} 个{word_type} 停用词")

    def remove_words(self, words: List[str]):
        if isinstance(words, str):
            words = [words]
        word_set = set(words)
        self.stopwords.difference_update(word_set)
        self.custom_stopwords.difference_update(word_set)
        self.domain_stopwords.difference_update(word_set)
        print(f"已移除 {len(word_set)} 个停用词")

    def update_word_stats(self, documents: List[List[str]]):
        for doc_id, words in enumerate(documents):
            word_counter = Counter(words)
            unique_words = set(words)
            for word in unique_words:
                self.word_stats[word]['doc_freq'] += 1
            for word, freq in word_counter.items():
                self.word_stats[word]['total_freq'] += freq
            for i, w in enumerate(words):
                context = words[max(0,i-2):i] + words[i+1:min(len(words), i+3)]
                self.word_stats[w]['contexts'].append(context)

    def discover_stopwords(self, documents: List[List[str]], freq_threshold: float=0.8, idf_threshold: float=0.1) -> Set[str]:
        if not self.word_stats:
            self.update_word_stats(documents)
        total_docs = len(documents)
        candidate_stopwords = set()
        for word, stats in self.word_stats.items():
            doc_freq_ratio = stats['doc_freq']/total_docs
            idf = np.log(total_docs/(stats['doc_freq']+1))
            if doc_freq_ratio >= freq_threshold and idf <= idf_threshold:
                candidate_stopwords.add(word)
        return candidate_stopwords

    def filter_stopwords(self, words: List[str], min_length: int=2) -> List[str]:
        return [word for word in words if word not in self.stopwords and len(word) >= min_length]

    def evaluate_stopwords_effect(self, documents: List[List[str]]) -> Dict:
        original_vocab = set()
        filtered_vocab = set()
        original_total_words = 0
        filtered_total_words = 0
        for words in documents:
            original_vocab.update(words)
            original_total_words += len(words)
            filtered_words = [w for w in words if w not in self.stopwords]
            filtered_vocab.update(filtered_words)
            filtered_total_words += len(filtered_words)
        vocab_compression = 1 - len(filtered_vocab)/len(original_vocab)
        text_compression = 1 - filtered_total_words/original_total_words
        return {
            'original_vocabulary_size': len(original_vocab),
            'filtered_vocabulary_size': len(filtered_vocab),
            'vocabulary_compression_ratio': vocab_compression,
            'original_total_words': original_total_words,
            'filtered_total_words': filtered_total_words,
            'text_compression_ratio': text_compression,
            'stopwords_count': len(self.stopwords)
        }

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/usr/local/lib/python3.12/dist-packages/jieba/posseg/__init__.py:16: SyntaxWarning: invalid escape sequence '\.'
  re_skip_detail = re.compile("([\.0-9]+|[a-zA-Z0-9]+)")
/usr/local/lib/python3.12/dist-packages/jieba/posseg/__init__.py:17: SyntaxWarning: invalid escape sequence '\.'
  re_han_internal = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._]+)")
/usr/local/lib/python3.12/dist-packages/jieba/posseg/__init__.py:18: SyntaxWarning: invalid escape sequence '\s'
  re_skip_internal = re.compil

In [ ]:
if __name__ == "__main__":
    # --- 测试中文分词器 ---
    tokenizer = AdvancedChineseTokenizer()
    text = "我爱自然语言处理技术和机器学习"
    print("精确模式:", tokenizer.precise_cut(text))
    print("全模式:", tokenizer.full_cut(text))
    print("搜索模式:", tokenizer.search_cut(text))
    print("词性标注:", tokenizer.pos_tagging(text))
    print("关键词(TF-IDF):", tokenizer.extract_keywords(text))

    # --- 测试智能停用词管理 ---
    stop_manager = IntelligentStopwordsManager()
    documents = [['我','爱','自然','语言','处理','技术'], ['机器','学习','是','人工智能','的','重要','分支']]
    stop_manager.update_word_stats(documents)
    discovered = stop_manager.discover_stopwords(documents, freq_threshold=0.5, idf_threshold=1.0)
    print("发现停用词候选:", discovered)
    filtered = stop_manager.filter_stopwords(['我','爱','自然','语言','处理','的','技术'])
    print("过滤后:", filtered)
    eval_res = stop_manager.evaluate_stopwords_effect(documents)
    print("停用词效果评估:", eval_res)

Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 1.643 seconds.
DEBUG:jieba:Loading model cost 1.643 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


精确模式: ['我', '爱', '自然语言', '处理', '技术', '和', '机器', '学习']
全模式: ['我', '爱', '自然', '自然语言', '语言', '处理', '技术', '和', '机器', '学习']
搜索模式: ['我', '爱', '自然', '语言', '自然语言', '处理', '技术', '和', '机器', '学习']
词性标注: [('我', 'r'), ('爱', 'v'), ('自然语言', 'l'), ('处理', 'v'), ('技术', 'n'), ('和', 'c'), ('机器', 'n'), ('学习', 'v')]
关键词(TF-IDF): [('自然语言', 2.08698834984), ('机器', 1.328312304776), ('学习', 1.155423963382), ('处理', 1.082171131472), ('技术', 0.943891435714)]
发现停用词候选: {'是', '我', '的', '自然', '重要', '爱', '语言', '处理', '技术', '机器', '分支', '人工智能', '学习'}
过滤后: ['自然', '语言', '处理', '技术']
停用词效果评估: {'original_vocabulary_size': 13, 'filtered_vocabulary_size': 10, 'vocabulary_compression_ratio': 0.23076923076923073, 'original_total_words': 13, 'filtered_total_words': 10, 'text_compression_ratio': 0.23076923076923073, 'stopwords_count': 11}
